# 02 — `Pipeline.from_zarr` end-to-end on a synthetic Zarr

Notebook 01 ran the four transforms stage-by-stage. The `Pipeline`
class chains all four with the intermediates kept on-device (no CSV /
binary disk round-trips between stages) and writes only the final
outputs on `dump()`.

This notebook builds a **synthetic `*.MIDAS.zip` Zarr archive** plus
the `AllPeaks_PS.bin` peak blob that the production peak-fitter would
emit, then runs `Pipeline.from_zarr(...).run()` and `.dump()`. CPU
only, a few seconds.


In [1]:
import math
import numpy as np
from midas_transforms.merge.core import (
    COL_SPOTID, COL_II, COL_OMEGA, COL_YCEN, COL_ZCEN, COL_IMAX,
    COL_RADIUS, COL_ETA, COL_SIGMAR, COL_SIGMAETA, COL_NRPX,
    COL_NRPXTOT, COL_RAWSUM, N_PEAK_COLS,
)

# Geometry shared across the notebook.
NR_PIXELS = 2048
PX_UM = 200.0          # pixel size
LSD_UM = 1_000_000.0   # sample-detector distance
YCEN = ZCEN = 1024.0   # beam center (px)
RINGS = [1, 2, 3]
RING_RADII_UM = [94949.4, 109761.7, 155930.4]   # ring radii in µm
N_FRAMES = 4
ETAS = [-120.0, -40.0, 40.0, 120.0]

def synth_frames():
    """One (n_peaks, 29) AllPeaks_PS-layout array per ω frame.

    Each peak is placed on its ring at the chosen η, so its observed
    radius matches the hkls ring radius and survives calc_radius.
    """
    frames = []
    sid = 1
    for fi in range(N_FRAMES):
        rows = []
        for rr in RING_RADII_UM:
            r_px = rr / PX_UM
            for eta in ETAS:
                y = YCEN - r_px * math.sin(math.radians(eta))
                z = ZCEN + r_px * math.cos(math.radians(eta))
                row = np.zeros(N_PEAK_COLS)
                row[COL_SPOTID] = sid; sid += 1
                row[COL_II] = 100.0
                row[COL_OMEGA] = fi * 0.25
                row[COL_YCEN] = y; row[COL_ZCEN] = z
                row[COL_IMAX] = 200.0
                row[COL_RADIUS] = r_px        # radial distance (px) → ring radius
                row[COL_ETA] = eta
                row[COL_SIGMAR] = 1.0; row[COL_SIGMAETA] = 1.0
                row[COL_NRPX] = 9; row[COL_NRPXTOT] = 9
                row[COL_RAWSUM] = 100.0
                rows.append(row)
        frames.append(np.array(rows))
    return frames

def write_hkls(path):
    with open(path, "w") as f:
        f.write("h k l D-spacing RingNr g1 g2 g3 Theta 2Theta Radius\n")
        for rn, rr in zip(RINGS, RING_RADII_UM):
            f.write(f"1 1 1 2.0 {rn} 0 0 0 1.0 2.0 {rr}\n")

import tempfile
from pathlib import Path
import zarr
import torch

WORK = Path(tempfile.mkdtemp(prefix="midas_transforms_pipe_"))
(WORK / "Temp").mkdir()
print("torch:", torch.__version__, "| device: cpu")
print("workspace:", WORK)


torch: 2.11.0 | device: cpu
workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_transforms_pipe_j70rha4a


## 1. Write a synthetic `*.MIDAS.zip`

A MIDAS analysis Zarr stores parameters under
`analysis/process/analysis_parameters`. We write the keys
`read_zarr_params` requires (geometry, ring thresholds, lattice,
margins). This is the same group `midas_zipper` produces from raw
detector data.


In [2]:
zip_fn = WORK / "synthFF_000001.analysis.MIDAS.zip"
store = zarr.ZipStore(str(zip_fn), mode="w")
root = zarr.group(store=store)
ap = root.create_group("analysis").create_group("process").create_group("analysis_parameters")

def put(key, value, dt):
    ap.create_dataset(key, data=np.array(value, dtype=dt))

for key, val in [
    ("Lsd", [LSD_UM]), ("Wavelength", [0.18]), ("PixelSize", [PX_UM]),
    ("YCen", [YCEN]), ("ZCen", [ZCEN]),
    ("tx", [0.0]), ("ty", [0.0]), ("tz", [0.0]),
    ("OmegaStart", [0.0]), ("OmegaStep", [0.25]),
    ("Width", [2000.0]), ("Hbeam", [2000.0]), ("Rsample", [1000.0]),
    ("MarginRadius", [2000.0]), ("MarginRadial", [2000.0]),
    ("MarginEta", [500.0]), ("MarginOme", [2.0]),
    ("EtaBinSize", [5.0]), ("OmeBinSize", [5.0]),
    ("StepSizeOrient", [0.2]), ("StepSizePos", [100.0]),
]:
    put(key, val, np.double)
put("LatticeParameter", [3.6, 3.6, 3.6, 90, 90, 90], np.double)
put("RingThresh", [[rn, 100.0] for rn in RINGS], np.double)
put("NrPixels", [NR_PIXELS], np.int32)
put("SpaceGroup", [225], np.int32)
store.close()

write_hkls(WORK / "hkls.csv")   # ring radii table the radius stage reads
print("wrote", zip_fn.name, f"({zip_fn.stat().st_size} bytes)")


wrote synthFF_000001.analysis.MIDAS.zip (17885 bytes)


## 2. Write the synthetic `AllPeaks_PS.bin`

`Pipeline` reads the consolidated peak blob the peak-fitter writes
(`Temp/AllPeaks_PS.bin`). Its layout is: `int32 n_frames`, then
per-frame `int32` peak counts, then per-frame `int64` byte offsets,
then the float64 `(n_peaks, 29)` blocks. We serialize our synthetic
frames into exactly that format.


In [3]:
frames = synth_frames()
counts = np.array([f.shape[0] for f in frames], dtype=np.int32)
n_frames = len(frames)
header = 4 + n_frames * 4 + n_frames * 8
offsets, off = [], header
for f in frames:
    offsets.append(off)
    off += f.shape[0] * N_PEAK_COLS * 8

ps_bin = WORK / "Temp" / "AllPeaks_PS.bin"
with open(ps_bin, "wb") as fh:
    fh.write(np.array([n_frames], dtype=np.int32).tobytes())
    fh.write(counts.tobytes())
    fh.write(np.array(offsets, dtype=np.int64).tobytes())
    for f in frames:
        fh.write(f.astype(np.float64).tobytes())
print("wrote", ps_bin.relative_to(WORK), f"({ps_bin.stat().st_size} bytes),"
      f" {int(counts.sum())} peaks across {n_frames} frames")


wrote Temp/AllPeaks_PS.bin (11188 bytes), 48 peaks across 4 frames


## 3. Build the pipeline and run all four stages

`Pipeline.from_zarr` parses the Zarr params and locates the
`AllPeaks_PS.bin`. `run()` executes merge → radius → fit-setup → bin
with the intermediates staying as on-device tensors. (We set `EndNr`
to our frame count, which a real scan's Zarr would carry.)


In [4]:
from midas_transforms import Pipeline

pipe = Pipeline.from_zarr(zip_fn, result_folder=WORK, device="cpu", dtype="float64")
pipe.zarr_params.EndNr = N_FRAMES
result = pipe.run()

print("merge      :", tuple(result.merge.peaks.shape), "(N, 17)")
print("radius     :", tuple(result.radius.spots.shape), "(N, 24)")
print("fit_setup  :", tuple(result.fit_setup.spots_inputall.shape), "(N, 8)")
print("bins/Spots :", tuple(result.bins.spots.shape))
print("bins/Data  :", tuple(result.bins.data.shape),
      "| nData:", tuple(result.bins.ndata.shape))


merge      : (12, 17) (N, 17)
radius     : (12, 24) (N, 24)
fit_setup  : (12, 8) (N, 8)
bins/Spots : (12, 10)
bins/Data  : (0,) | nData: (15552, 2)


## 4. Dump the outputs

`dump(out_dir)` writes every intermediate + final file the per-stage
CLIs would have produced — including `Spots.bin` / `Data.bin` /
`nData.bin` and `SpotsToIndex.csv`, the exact handoff to
`midas-index`.


In [5]:
out_dir = WORK / "out"
pipe.dump(out_dir)
print("dumped files:")
for p in sorted(out_dir.iterdir()):
    print(f"  {p.name}")


dumped files:
  Data.bin
  ExtraInfo.bin
  InputAll.csv
  InputAllExtraInfoFittingAll.csv
  MergeMap.csv
  Radius_StartNr_1_EndNr_4.csv
  Result_StartNr_1_EndNr_4.csv
  Spots.bin
  SpotsToIndex.csv
  nData.bin
  paramstest.txt
  positions.csv


## Recap

- `Pipeline.from_zarr(zarr).run()` chains all four transforms with
  on-device intermediates; `dump()` writes the indexer-ready files.
- The synthetic `*.MIDAS.zip` here mirrors what `midas_zipper`
  produces from raw frames; the `AllPeaks_PS.bin` mirrors what
  `midas-peakfit` produces.
- On a GPU box, pass `device="cuda"` to `from_zarr` — the same code
  path, no `.cu` sources. (This machine is CPU-only.)
- CLI equivalent: `midas-transforms pipeline scan.zip --out-dir …`.
